# Current 32-feature ranker export

Export the current direction-aware ranker without changing the earlier `exports/final_model.zip`. This is a provisional ranker handoff, not a completed five-model selection: LR, KNN and the final stacker are still pending integration.

The current student-test Top-1 is 89.8904%. These trips have been used for development comparisons; this is not a new independent evaluator test. The current optional HMM is included but disabled by default.

In [1]:
from pathlib import Path
import json
import shutil
import hashlib
import zipfile
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
import nbformat
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb

source = Path("results/direction_features")
bundle = Path("exports/current_ranker")
bundle.mkdir(parents=True, exist_ok=True)
old_archive = Path("exports/final_model.zip")
old_hash = hashlib.sha256(old_archive.read_bytes()).hexdigest() if old_archive.exists() else None
features = json.loads((source / "feature_columns.json").read_text())
training = json.loads((source / "config.json").read_text())
metrics = json.loads((source / "student_test_metrics.json").read_text())
assert len(features) == len(set(features)) == 32
assert training["excluded_trip_ids"] == ["00000033", "00000054"]
shutil.copy2(source / "model.json", bundle / "ranker.json")
shutil.copy2(source / "feature_columns.json", bundle / "feature_columns.json")
shutil.copy2("outputs/training_exclusions.json", bundle / "training_exclusions.json")

PosixPath('exports/current_ranker/training_exclusions.json')

## Frozen inference settings and definitions

Keep the raw longitude/latitude order, per-trip UTM projection, candidate selection and feature definitions consistent with training. Do not apply the reference-derived GPS quality flag at inference; that threshold is evaluation-only. Model input contains no reference route or labels.

In [2]:
candidate = training["candidate_settings"]
(bundle / "candidate_settings.json").write_text(json.dumps({"radius_m": candidate["radius_m"], "K": candidate["K"],
    "sort_order": ["group_id", "distance", "segment_id"], "segment_ids": "zero-based arc row IDs"}, indent=2) + "\n")
direction = {name: training[name] for name in ["half_windows_seconds", "heading_min_speed_mps"] if name in training}
direction["heading_min_displacement_m"] = training["heading_min_displacement_m"]
direction["added_features"] = training["added_features"]
direction["feature_version"] = "direction_32"
(bundle / "direction_config.json").write_text(json.dumps(direction, indent=2) + "\n")
preprocessing = {"coordinate_order": ["longitude", "latitude"], "coordinate_system": "WGS84 to per-trip UTM, always_xy=True",
    "utm_zone": "int((mean_lon+180)//6)+1; EPSG=(32600 north or 32700 south)+zone",
    "feature_dtype": "float32", "missing_values": "NaN, handled natively by XGBoost", "stationary_mps": 0.5,
    "context_point_offsets": [-2, -1, 1, 2], "request_mode": "whole trip; future observations required",
    "distance_ratio": "(distance+1)/(distance-distance_above_nearest_m+1)",
    "probabilities": "stable softmax within each GPS group", "temperature": 1.0,
    "reference_route_required": False, "reference_quality_filter_at_inference": False}
(bundle / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2) + "\n")
hmm = json.loads(Path("results/hmm/direction_ranker/hmm_config.json").read_text())
assert hmm["feature_count"] == 32
hmm["enabled"] = False
hmm["availability"] = "Optional current-ranker decoder; disabled by default because classifier-only student-test Top-1 is higher."
(bundle / "hmm_config.json").write_text(json.dumps(hmm, indent=2) + "\n")
helpers = ["05_feature_helpers.ipynb", "05_direction_features.ipynb", "08_hmm_helpers.ipynb"]
for filename in helpers:
    helper = nbformat.read(filename, as_version=4)
    for cell in helper.cells:
        if cell.cell_type == "code":
            cell.outputs = []
            cell.execution_count = None
    nbformat.write(helper, bundle / filename)
packages = ["numpy", "pandas", "shapely", "pyproj", "pyarrow", "scipy", "xgboost", "nbformat"]
versions = {name: version(name) for name in packages}
(bundle / "requirements.txt").write_text("\n".join(f"{name}=={value}" for name, value in versions.items()) + "\n")
(bundle / "README.md").write_text("""# Current PathSnap ranker

This provisional bundle uses 32 ordered features and excludes training trips 00000033 and 00000054.
Load ranker.json with XGBRanker.load_model and select feature_columns.json in its exact order.
Generate candidates from raw nodes/arcs/track using candidate_settings.json, then use the included feature and direction notebooks plus preprocessing.json.
Apply stable per-point softmax. The optional current-model HMM is configured in hmm_config.json but enabled=false by default.
The helpers require only observed GPS and road information. Do not pass reference routes or point labels as predictor inputs.
Student-test Top-1 is recorded in manifest.json as development evidence, not an independent final evaluator score.
This is an artifact bundle; API integration is in repository notebook 13_backend_api.ipynb. No running server or front end is included.
""")

881

## Reload and equality check

Read a complete candidate group from the feature-only direction table. Neither labels nor reference metadata are loaded. The exported model must retain all 32 names, match source predictions exactly and produce finite normalized probabilities.

In [3]:
sample = next(pq.ParquetFile("outputs/direction_features/student_test.parquet").iter_batches(
    batch_size=256, columns=["group_id"] + features)).to_pandas()
sample = sample[sample["group_id"] == sample["group_id"].iloc[0]].copy()
assert len(sample) == int(sample["candidate_count"].iloc[0])
model = xgb.XGBRanker(n_jobs=2)
model.load_model(bundle / "ranker.json")
original = xgb.XGBRanker(n_jobs=2)
original.load_model(source / "model.json")
assert model.get_booster().feature_names == features
raw = model.predict(sample[features])
np.testing.assert_allclose(raw, original.predict(sample[features]), atol=0, rtol=0)
assert np.isfinite(raw).all()
probability = np.exp(raw.astype(float) - raw.max())
probability /= probability.sum()
np.testing.assert_allclose(probability.sum(), 1, atol=1e-12)
validation = {"feature_count": 32, "model_reloaded": True, "source_predictions_equal": True,
    "sample_rows": len(sample), "complete_query": True, "finite_scores": True, "normalized_probabilities": True}

## Portable manifest and versioned archive

Only small model/config/helper artifacts are included. Huge OOF tables, feature tables, raw files and reference routes are omitted. The older `final_model.zip` remains unchanged.

In [4]:
files = ["ranker.json", "feature_columns.json", "candidate_settings.json", "direction_config.json", "preprocessing.json",
         "training_exclusions.json", "hmm_config.json", "requirements.txt", "README.md"] + helpers
hashes = {name: hashlib.sha256((bundle / name).read_bytes()).hexdigest() for name in files}
manifest = {"created_at": datetime.now(ZoneInfo("Asia/Kolkata")).isoformat(), "model": "xgboost_ranker",
    "feature_version": "direction_32", "feature_count": len(features), "hmm_enabled_by_default": False,
    "selection_status": "provisional current ranker; final LR/KNN/decision-tree/stacking comparison pending consolidation",
    "excluded_trip_ids": training["excluded_trip_ids"], "student_test_metrics": metrics,
    "evaluation_scope": "repeated student-test development comparison; not fresh independent evaluator test",
    "versions": versions, "artifacts": hashes, "validation": validation, "api_status": "artifact export; no LAN deployment claimed"}
(bundle / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
archive = bundle.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
    for filename in files + ["manifest.json"]:
        assert not Path(filename).is_absolute()
        zipped.write(bundle / filename, arcname=filename)
with zipfile.ZipFile(archive) as zipped:
    assert zipped.testzip() is None
    assert set(zipped.namelist()) == set(files + ["manifest.json"])
    for filename, digest in hashes.items():
        assert hashlib.sha256(zipped.read(filename)).hexdigest() == digest
if old_hash is not None:
    assert hashlib.sha256(old_archive.read_bytes()).hexdigest() == old_hash
print("Exported:", archive, "| Student-test Top-1:", metrics["top1"], "| 32 features | HMM default disabled")
print("Model reload/equality, checksums, archive integrity and preservation of the older bundle passed.")

Exported: exports/current_ranker.zip | Student-test Top-1: 0.8989039665970773 | 32 features | HMM default disabled
Model reload/equality, checksums, archive integrity and preservation of the older bundle passed.


## Optional newly integrated DT API smoke

If DT integration has completed by this point, verify its real backend request. Otherwise report its pending state without delaying this ranker export.

In [5]:
registry = json.loads(Path("models/registry.json").read_text())
entry = registry["models"]["decision_tree"]
if entry.get("status") == "trained" and Path(entry.get("model_path", "")).is_file():
    import runpy
    from fastapi.testclient import TestClient
    api = runpy.run_path(".runtime/api.py")
    response = TestClient(api["app"]).post("/predict", json={"trip_id": "00000052", "model": "decision_tree"})
    assert response.status_code == 200, response.text
    result = response.json()
    assert result["model"] == "decision_tree" and len(result["predictions"]) == result["point_count"]
    serialized = json.dumps(result)
    assert all(f'"{name}"' not in serialized for name in ["true_arc_id", "route_pos", "label", "bad_gps", "reference_route"])
    print("Real DT API smoke passed:", result["point_count"], "predictions.")
else:
    print("DT API smoke pending: registry is not yet trained; current ranker export is complete.")

DT API smoke pending: registry is not yet trained; current ranker export is complete.
